In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

In [2]:
import sklearn
print(sklearn.__version__)

1.9.0


# Data Loading and Dataset Checking

In [3]:
df=pd.read_csv("Restaurant_feature_engineered.csv")
df.head()


,CuisineType,RestaurantID,RestaurantName,Segment,Subregion,GrowthFactor,AOV,MonthlyOrders,InStoreOrders,InStoreRevenue,...,DD_share,SD_share,TotalRevenue,TotalProfit,ProfitMargin,AggregatorDependence,CostBurden,RevenueQuality,Raw_Scale_Signal,Expansion_Headroom
0,Burgers,25731,Urban Burgers House,Cafe,North Shore,1.03,43.97,668,197,8662.09,...,0.25,0.3,29372.0,7965.0,0.271177,0.7,0.574913,11.923636,688.04,11.33
1,Burgers,25123,Urban Burgers Diner,QSR,South Auckland,1.05,40.45,1388,259,10476.55,...,0.25,0.3,56145.0,8776.0,0.156310,0.7,0.655830,6.322722,1457.40,16.80
2,Burgers,25177,King Burgers Eatery,Cafe,West Auckland,1.04,40.03,1717,524,20975.72,...,0.25,0.3,68732.0,14403.0,0.209553,0.7,0.627620,8.388408,1785.68,11.44
3,Burgers,25540,Classic Burgers Tavern,QSR,North Shore,1.03,36.28,1083,216,7836.48,...,0.25,0.3,39291.0,5267.0,0.134051,0.7,0.675106,4.863372,1115.49,3.09
4,Burgers,25258,Lucky Burgers Bistro,Cafe,South Auckland,1.05,34.34,1230,261,8962.74,...,0.25,0.3,42238.0,6119.0,0.144870,0.7,0.654894,4.974820,1291.50,9.45


In [4]:
df.shape
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1696 entries, 0 to 1695
Data columns (total 38 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   CuisineType            1696 non-null   str    
 1   RestaurantID           1696 non-null   int64  
 2   RestaurantName         1696 non-null   str    
 3   Segment                1696 non-null   str    
 4   Subregion              1696 non-null   str    
 5   GrowthFactor           1696 non-null   float64
 6   AOV                    1696 non-null   float64
 7   MonthlyOrders          1696 non-null   int64  
 8   InStoreOrders          1696 non-null   int64  
 9   InStoreRevenue         1696 non-null   float64
 10  UberEatsOrders         1696 non-null   int64  
 11  DoorDashOrders         1696 non-null   int64  
 12  SelfDeliveryOrders     1696 non-null   int64  
 13  UberEatsRevenue        1696 non-null   float64
 14  DoorDashRevenue        1696 non-null   float64
 15  SelfDeliveryRev

# Modeling Features : Drop Columns

In [5]:
drop_cols=['RestaurantID','RestaurantName']
df_model= df.drop(columns=drop_cols)
df_model

,CuisineType,Segment,Subregion,GrowthFactor,AOV,MonthlyOrders,InStoreOrders,InStoreRevenue,UberEatsOrders,DoorDashOrders,...,DD_share,SD_share,TotalRevenue,TotalProfit,ProfitMargin,AggregatorDependence,CostBurden,RevenueQuality,Raw_Scale_Signal,Expansion_Headroom
0,Burgers,Cafe,North Shore,1.03,43.97,668,197,8662.09,212,118,...,0.25,0.3,29372.0,7965.0,0.271177,0.7,0.574913,11.923636,688.04,11.33
1,Burgers,QSR,South Auckland,1.05,40.45,1388,259,10476.55,508,282,...,0.25,0.3,56145.0,8776.0,0.156310,0.7,0.655830,6.322722,1457.40,16.80
2,Burgers,Cafe,West Auckland,1.04,40.03,1717,524,20975.72,537,298,...,0.25,0.3,68732.0,14403.0,0.209553,0.7,0.627620,8.388408,1785.68,11.44
3,Burgers,QSR,North Shore,1.03,36.28,1083,216,7836.48,390,217,...,0.25,0.3,39291.0,5267.0,0.134051,0.7,0.675106,4.863372,1115.49,3.09
4,Burgers,Cafe,South Auckland,1.05,34.34,1230,261,8962.74,436,242,...,0.25,0.3,42238.0,6119.0,0.144870,0.7,0.654894,4.974820,1291.50,9.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1691,Thai,Ghost Kitchen,North Shore,1.03,45.81,487,31,1420.11,228,137,...,0.30,0.2,22309.0,6254.0,0.280335,0.8,0.474365,12.842160,501.61,17.51
1692,Thai,Ghost Kitchen,North Shore,1.03,44.20,956,62,2740.40,447,268,...,0.30,0.2,42255.0,10253.0,0.242646,0.8,0.511706,10.724946,984.68,17.51
1693,Thai,QSR,South Auckland,1.05,44.72,758,62,2772.64,348,209,...,0.30,0.2,33898.0,3671.0,0.108295,0.8,0.650500,4.842974,795.90,5.25
1694,Thai,Cafe,North Shore,1.03,36.98,1084,124,4585.52,480,288,...,0.30,0.2,40086.0,5626.0,0.140348,0.8,0.654190,5.190078,1116.52,5.15


# Feature Separation

In [6]:
num_cols= df_model.select_dtypes(
    include=['int64','float64']
).columns
cat_cols=df_model.select_dtypes(include='object').columns
print('Numerical Features:')
print(list(num_cols))

print('\nCategorical Features:')
print(list(cat_cols))


Numerical Features:
['GrowthFactor', 'AOV', 'MonthlyOrders', 'InStoreOrders', 'InStoreRevenue', 'UberEatsOrders', 'DoorDashOrders', 'SelfDeliveryOrders', 'UberEatsRevenue', 'DoorDashRevenue', 'SelfDeliveryRevenue', 'COGSRate', 'OPEXRate', 'CommissionRate', 'DeliveryRadiusKM', 'DeliveryCostPerOrder', 'SD_DeliveryTotalCost', 'InStoreNetProfit', 'UberEatsNetProfit', 'DoorDashNetProfit', 'SelfDeliveryNetProfit', 'InStoreShare', 'UE_share', 'DD_share', 'SD_share', 'TotalRevenue', 'TotalProfit', 'ProfitMargin', 'AggregatorDependence', 'CostBurden', 'RevenueQuality', 'Raw_Scale_Signal', 'Expansion_Headroom']

Categorical Features:
['CuisineType', 'Segment', 'Subregion']


C:\Users\malin\AppData\Local\Temp\ipykernel_23484\2826042901.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols=df_model.select_dtypes(include='object').columns


# Skewness Checking

In [7]:
skew_df= df_model[num_cols].skew().sort_values(ascending=False)
skew_df

InStoreNetProfit         1.645656
SelfDeliveryNetProfit    1.492219
SD_DeliveryTotalCost     1.387257
SelfDeliveryRevenue      1.301389
SelfDeliveryOrders       1.161952
InStoreRevenue           1.136530
InStoreOrders            1.063820
SD_share                 0.976599
COGSRate                 0.698871
CostBurden               0.655182
InStoreShare             0.519725
UberEatsRevenue          0.422275
DoorDashRevenue          0.416967
TotalRevenue             0.354974
UberEatsOrders           0.272789
DoorDashOrders           0.253951
Raw_Scale_Signal         0.206973
MonthlyOrders            0.191067
TotalProfit              0.053253
AOV                      0.019457
Expansion_Headroom      -0.034636
DeliveryRadiusKM        -0.040665
DeliveryCostPerOrder    -0.040711
CommissionRate          -0.072960
RevenueQuality          -0.448785
DD_share                -0.476435
OPEXRate                -0.478374
UE_share                -0.552063
ProfitMargin            -0.570472
DoorDashNetPro

In [8]:
skew_df.to_frame(name='Skewness')

,Skewness
InStoreNetProfit,1.645656
SelfDeliveryNetProfit,1.492219
SD_DeliveryTotalCost,1.387257
SelfDeliveryRevenue,1.301389
SelfDeliveryOrders,1.161952
InStoreRevenue,1.136530
InStoreOrders,1.063820
SD_share,0.976599
COGSRate,0.698871
CostBurden,0.655182


# Creation of Transformed Version

In [9]:
df['Log_SD_DeliveryTotalCost']=np.log1p(
    df['SD_DeliveryTotalCost']
)
df['Log_SD_DeliveryTotalCost'].skew()

np.float64(-0.21325600662537775)

In [10]:
df=df.drop(columns=['SD_DeliveryTotalCost'])

# Checking of Categorical Variables

In [11]:
print(df['CuisineType'].nunique())
print(df['Segment'].nunique())
print(df['Subregion'].nunique())

8
4
4


In [12]:
print(df['CuisineType'].unique())
print(df['Segment'].unique())
print(df['Subregion'].unique())

<ArrowStringArray>
[             'Burgers',       'Chicken Dishes',              'Chinese',
               'Indian',             'Japanese', 'Kebabs/Mediterranean',
                'Pizza',                 'Thai']
Length: 8, dtype: str
<ArrowStringArray>
['Cafe', 'QSR', 'Ghost Kitchen', 'Full-service']
Length: 4, dtype: str
<ArrowStringArray>
['North Shore', 'South Auckland', 'West Auckland', 'CBD']
Length: 4, dtype: str


# Application of One-Hot-Encoding

In [16]:
df_encoded=pd.get_dummies(
  df_model,
  columns=['CuisineType','Segment','Subregion'],
  drop_first= True
)

In [17]:
df_encoded.shape

(1696, 46)

In [18]:
df_encoded.head()

,GrowthFactor,AOV,MonthlyOrders,InStoreOrders,InStoreRevenue,UberEatsOrders,DoorDashOrders,SelfDeliveryOrders,UberEatsRevenue,DoorDashRevenue,...,CuisineType_Japanese,CuisineType_Kebabs/Mediterranean,CuisineType_Pizza,CuisineType_Thai,Segment_Full-service,Segment_Ghost Kitchen,Segment_QSR,Subregion_North Shore,Subregion_South Auckland,Subregion_West Auckland
0,1.03,43.97,668,197,8662.09,212,118,141,9321.64,5188.46,...,False,False,False,False,False,False,False,True,False,False
1,1.05,40.45,1388,259,10476.55,508,282,339,20548.60,11406.90,...,False,False,False,False,False,False,True,False,True,False
2,1.04,40.03,1717,524,20975.72,537,298,358,21496.11,11928.94,...,False,False,False,False,False,False,False,False,False,True
3,1.03,36.28,1083,216,7836.48,390,217,260,14149.20,7872.76,...,False,False,False,False,False,False,True,True,False,False
4,1.05,34.34,1230,261,8962.74,436,242,291,14972.24,8310.28,...,False,False,False,False,False,False,False,False,True,False


## Object Columns Checking

In [19]:
df_encoded.select_dtypes(include='object').columns

Index([], dtype='str')

In [22]:
df_encoded.shape


(1696, 46)

In [23]:
df_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 1696 entries, 0 to 1695
Data columns (total 46 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   GrowthFactor                      1696 non-null   float64
 1   AOV                               1696 non-null   float64
 2   MonthlyOrders                     1696 non-null   int64  
 3   InStoreOrders                     1696 non-null   int64  
 4   InStoreRevenue                    1696 non-null   float64
 5   UberEatsOrders                    1696 non-null   int64  
 6   DoorDashOrders                    1696 non-null   int64  
 7   SelfDeliveryOrders                1696 non-null   int64  
 8   UberEatsRevenue                   1696 non-null   float64
 9   DoorDashRevenue                   1696 non-null   float64
 10  SelfDeliveryRevenue               1696 non-null   float64
 11  COGSRate                          1696 non-null   float64
 12  OPEXRate         

# Feature Scaling

In [24]:
scaler= StandardScaler()
df_scaled= pd.DataFrame(
    scaler.fit_transform(df_encoded),
    columns=df_encoded.columns
)

In [26]:
df_scaled.head()

,GrowthFactor,AOV,MonthlyOrders,InStoreOrders,InStoreRevenue,UberEatsOrders,DoorDashOrders,SelfDeliveryOrders,UberEatsRevenue,DoorDashRevenue,...,CuisineType_Japanese,CuisineType_Kebabs/Mediterranean,CuisineType_Pizza,CuisineType_Thai,Segment_Full-service,Segment_Ghost Kitchen,Segment_QSR,Subregion_North Shore,Subregion_South Auckland,Subregion_West Auckland
0,0.076907,1.223194,-1.237599,-0.146816,0.040977,-1.513447,-1.503112,-0.781182,-1.271834,-1.256592,...,-0.275939,-0.330487,-0.385063,-0.275939,-0.60278,-0.36252,-0.686255,1.851539,-0.592785,-0.611876
1,0.958215,0.433619,0.467712,0.267305,0.351168,0.210176,0.268368,0.742922,0.341887,0.402109,...,-0.275939,-0.330487,-0.385063,-0.275939,-0.60278,-0.36252,1.457185,-0.540091,1.686951,-0.611876
2,0.517561,0.339408,1.246945,2.037341,2.146048,0.379044,0.441195,0.889174,0.478078,0.541358,...,-0.275939,-0.330487,-0.385063,-0.275939,-0.60278,-0.36252,-0.686255,-0.540091,-0.592785,1.634318
3,0.076907,-0.501759,-0.254677,-0.019908,-0.100164,-0.476944,-0.433743,0.134820,-0.577939,-0.540584,...,-0.275939,-0.330487,-0.385063,-0.275939,-0.60278,-0.36252,1.457185,1.851539,-0.592785,-0.611876
4,0.958215,-0.936922,0.093491,0.280664,0.092375,-0.209084,-0.163701,0.373442,-0.459638,-0.423880,...,-0.275939,-0.330487,-0.385063,-0.275939,-0.60278,-0.36252,-0.686255,-0.540091,1.686951,-0.611876


In [27]:
df_scaled.describe().round(2)

,GrowthFactor,AOV,MonthlyOrders,InStoreOrders,InStoreRevenue,UberEatsOrders,DoorDashOrders,SelfDeliveryOrders,UberEatsRevenue,DoorDashRevenue,...,CuisineType_Japanese,CuisineType_Kebabs/Mediterranean,CuisineType_Pizza,CuisineType_Thai,Segment_Full-service,Segment_Ghost Kitchen,Segment_QSR,Subregion_North Shore,Subregion_South Auckland,Subregion_West Auckland
count,1696.00,1696.00,1696.00,1696.00,1696.00,1696.00,1696.00,1696.00,1696.00,1696.00,...,1696.00,1696.00,1696.00,1696.00,1696.00,1696.00,1696.00,1696.00,1696.00,1696.00
mean,0.00,-0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,...,0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00
std,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,...,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00
min,-1.69,-1.96,-1.78,-1.38,-1.37,-1.92,-1.90,-1.40,-1.89,-1.87,...,-0.28,-0.33,-0.39,-0.28,-0.60,-0.36,-0.69,-0.54,-0.59,-0.61
25%,0.08,-0.84,-0.86,-0.79,-0.77,-0.82,-0.84,-0.74,-0.81,-0.82,...,-0.28,-0.33,-0.39,-0.28,-0.60,-0.36,-0.69,-0.54,-0.59,-0.61
50%,0.52,-0.04,-0.02,-0.27,-0.28,-0.09,-0.09,-0.23,-0.09,-0.09,...,-0.28,-0.33,-0.39,-0.28,-0.60,-0.36,-0.69,-0.54,-0.59,-0.61
75%,0.96,0.89,0.79,0.59,0.53,0.76,0.75,0.49,0.70,0.73,...,-0.28,-0.33,-0.39,-0.28,1.66,-0.36,1.46,-0.54,1.69,1.63
max,0.96,1.95,2.72,4.01,4.60,2.71,2.26,3.54,3.57,3.04,...,3.62,3.03,2.60,3.62,1.66,2.76,1.46,1.85,1.69,1.63


In [28]:
df_scaled.to_csv('restaurant_preprocessed.csv',index=False)